upgradeing the miner to pull the number of the changes to each style in dwelling period

# Step 1: Clone

In [ ]:
from __future__ import annotations
import csv, os, re, subprocess, time, json
from pathlib import Path
from typing import Optional, List
import os
from pathlib import Path
from dotenv import load_dotenv
import requests  # make sure 'requests' is installed: pip install requests

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"
CLONE_ROOT   = WORK_ROOT / "clonesV1.0"
MANIFEST_CSV = WORK_ROOT / "clones_manifestV1.0.csv"

# New: where to store GitHub metadata (PRs + issues)
META_ROOT = WORK_ROOT / "metadataV1.0"

WITH_SUBMODULES   = False     # set True if you want submodules initialized/updated
WITH_LFS          = False     # set True if you need Git LFS objects
FETCH_PR_REFS     = True      # set False to skip GitHub PR heads
FETCH_GH_METADATA = True      # set False if you only want clones (no GitHub API calls)


# Path to your env file
ENV_FILE = r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"

# === LOAD .env ===
load_dotenv(ENV_FILE)

# Load GitHub tokens: GITHUB_TOKEN_1 ... GITHUB_TOKEN_6
TOKENS = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]

if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_Tokens.env (API metadata might be rate limited).")
else:
    print(f"ℹ️ Loaded {len(TOKENS)} GitHub token(s) from All_Tokens.env")
    print("Token lengths:", [len(t) for t in TOKENS])

# index of the current token (0-based)
token_index = 0

GITHUB_API_BASE = "https://api.github.com"


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
DEFAULT_TIMEOUT = 1800  # 30 minutes for huge repos
GIT_ENV = {"GIT_TERMINAL_PROMPT": "0", "GIT_ASKPASS": ""}


def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True,
       capture: bool = True, timeout: Optional[int] = DEFAULT_TIMEOUT) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env.update(GIT_ENV)
    return subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        capture_output=capture,
        text=True,
        timeout=timeout,
        env=env,
    )


def sh_ok(cmd: List[str], cwd: Optional[Path] = None, timeout: Optional[int] = DEFAULT_TIMEOUT) -> str:
    cp = sh(cmd, cwd=cwd, check=True, capture=True, timeout=timeout)
    return cp.stdout


def repo_dir_name_from_url(url: str) -> str:
    u = url.strip()
    # SSH form: git@host:owner/repo(.git)
    m = re.match(r"^[^@]+@([^:]+):([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        owner, name = m.group(2), m.group(3)
        return f"{owner}__{name}"
    # https://host/owner/repo(.git)
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u
    parts = [p for p in base.split("/") if p]
    if len(parts) >= 2:
        owner, name = parts[-2], parts[-1].removesuffix(".git")
        return f"{owner}__{name}"
    return base.replace("/", "__").removesuffix(".git")


def parse_github_owner_repo(url: str) -> Optional[tuple[str, str]]:
    """
    Return (owner, repo) for GitHub URLs, or None if not GitHub.
    Supports https://github.com/owner/repo(.git) and git@github.com:owner/repo(.git).
    """
    u = url.strip()
    # SSH form
    m = re.match(r"^git@github\.com:([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        return m.group(1), m.group(2)

    # HTTPS form
    if "github.com" not in u.lower():
        return None

    # Drop protocol
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u

    # Remove possible query/fragments
    base = base.split("?", 1)[0].split("#", 1)[0]

    parts = [p for p in base.split("/") if p]
    # parts like ["github.com", "owner", "repo(.git)"]
    if len(parts) >= 3 and parts[0].lower().startswith("github.com"):
        owner = parts[1]
        repo = parts[2].removesuffix(".git")
        return owner, repo

    return None


def github_get(path: str, params: Optional[dict] = None) -> list:
    """
    Basic GitHub API GET with pagination.
    Uses round-robin token rotation when hitting rate limits.
    Returns a list of items.
    """
    global token_index

    url = f"{GITHUB_API_BASE}{path}"
    items: list = []
    page = 1

    while True:
        # Build headers with the current token
        headers = {"Accept": "application/vnd.github+json"}
        current_token = TOKENS[token_index] if TOKENS else None
        if current_token:
            headers["Authorization"] = f"Bearer {current_token}"

        q = dict(params or {})
        q.setdefault("per_page", 100)
        q["page"] = page

        resp = requests.get(url, headers=headers, params=q)

        # Detect rate limit
        remaining = resp.headers.get("X-RateLimit-Remaining")
        is_rate_limited = (
            resp.status_code == 403
            and ("rate limit" in resp.text.lower() or remaining == "0")
        )

        if is_rate_limited:
            print(
                f"[rate limit] {path} page={page} with token index {token_index}. "
                f"Remaining={remaining}"
            )
            if TOKENS and len(TOKENS) > 1:
                old_index = token_index
                token_index = (token_index + 1) % len(TOKENS)
                print(f"  -> switching token {old_index} -> {token_index} and retrying...")
                continue
            else:
                print("  -> no alternative tokens; stopping.")
                break

        # If we reach here, it's not a rate-limit error
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break

        if isinstance(data, list):
            items.extend(data)
        else:
            # some endpoints return an object, not a list
            items.append(data)
            break

        if len(data) < q["per_page"]:
            # last page
            break

        page += 1

    return items




def fetch_github_prs_and_issues(owner: str, repo: str, out_dir: Path) -> None:
    """
    Download all PRs (with comments & reviews) and all issues (non-PR)
    and store them as JSON files in out_dir.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # base name: owner__repo
    base_name = f"{owner}__{repo}"

    print(f"  [meta] Fetching PRs for {owner}/{repo} ...")
    prs = github_get(f"/repos/{owner}/{repo}/pulls", params={"state": "all"})
    for pr in prs:
        number = pr.get("number")
        if number is None:
            continue
        # issue-style comments on the PR
        issue_comments = github_get(
            f"/repos/{owner}/{repo}/issues/{number}/comments"
        )
        # review comments on specific lines
        review_comments = github_get(
            f"/repos/{owner}/{repo}/pulls/{number}/comments"
        )
        # review events (approve/request-changes etc.)
        reviews = github_get(
            f"/repos/{owner}/{repo}/pulls/{number}/reviews"
        )
        pr["issue_comments"] = issue_comments
        pr["review_comments"] = review_comments
        pr["reviews"] = reviews

    # Save as: metadataV1.0/owner__repo_PRs.json
    with (out_dir / f"{base_name}_PRs.json").open("w", encoding="utf-8") as f:
        json.dump(prs, f, ensure_ascii=False, indent=2)

    print(f"  [meta] Fetching issues for {owner}/{repo} ...")
    issues = github_get(f"/repos/{owner}/{repo}/issues", params={"state": "all"})
    # Filter out PRs (issues endpoint includes PRs with a 'pull_request' field)
    pure_issues = [it for it in issues if "pull_request" not in it]

    for issue in pure_issues:
        number = issue.get("number")
        if number is None:
            continue
        comments = github_get(
            f"/repos/{owner}/{repo}/issues/{number}/comments"
        )
        issue["comments"] = comments

    # Save as: metadataV1.0/owner__repo_Issues.json
    with (out_dir / f"{base_name}_Issues.json").open("w", encoding="utf-8") as f:
        json.dump(pure_issues, f, ensure_ascii=False, indent=2)



def ensure_full_clone(
    url: str,
    dest_root: Path,
    with_submodules: bool = False,
    with_lfs: bool = False,
    fetch_pr_refs: bool = True,
) -> Path:
    """
    Ensures a non-shallow clone with full history of all branches & tags.
    Optionally fetches GitHub PR heads to refs/remotes/origin/pr/*,
    initializes submodules, and fetches LFS objects.
    """
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)

    if not (d.exists() and (d / ".git").exists()):
        # no-recurse-submodules avoids submodule cost unless requested later
        sh(
            ["git", "clone", "--no-recurse-submodules", "--tags", url, str(d)],
            capture=False,
        )
    else:
        # If repo exists, ensure origin URL is correct
        sh(["git", "remote", "set-url", "origin", url], cwd=d, check=False)

    # If shallow, unshallow
    cp = sh(["git", "rev-parse", "--is-shallow-repository"], cwd=d, check=False)
    if cp.returncode == 0 and cp.stdout.strip() == "true":
        sh(["git", "fetch", "--unshallow", "--tags"], cwd=d, capture=False)
    else:
        # Make sure we have tags even if already full
        sh(["git", "fetch", "--tags"], cwd=d, check=False, capture=False)

    # Fetch all branches under refs/heads/* and prune deleted ones
    sh(
        [
            "git",
            "fetch",
            "origin",
            "--prune",
            "--tags",
            "+refs/heads/*:refs/remotes/origin/*",
            "--quiet",
        ],
        cwd=d,
        check=False,
        capture=False,
    )

    # Best-effort PR refs (GitHub); harmless if not present
    if fetch_pr_refs:
        sh(
            [
                "git",
                "fetch",
                "origin",
                "+refs/pull/*/head:refs/remotes/origin/pr/*",
                "--quiet",
            ],
            cwd=d,
            check=False,
            capture=False,
        )

    # Optional: submodules
    if with_submodules:
        sh(
            ["git", "submodule", "update", "--init", "--recursive"],
            cwd=d,
            capture=False,
        )

    # Optional: LFS
    if with_lfs:
        # If git-lfs isn't installed, these will fail harmlessly due to check=False
        sh(["git", "lfs", "install"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "fetch", "--all"], cwd=d, check=False, capture=False)
        sh(["git", "lfs", "checkout"], cwd=d, check=False, capture=False)

    return d


def get_total_commits(repo_dir: Path) -> int:
    # Count across all refs reachable in the repository
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir, check=False)
    try:
        return int((cp.stdout or "0").strip() or "0")
    except Exception:
        return 0


# -----------------------------
# Main: FULL CLONE + GH METADATA
# -----------------------------
def main() -> None:
    assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

    rows, ok, fail = [], 0, 0
    with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            url = (row.get("repo_url") or "").strip()
            if not url:
                continue
            t0 = time.time()
            rec = {
                "repo_url": url,
                "dir": None,
                "status": "unknown",
                "seconds": None,
                "total_commits": None,
                "error": "",
            }
            try:
                d = ensure_full_clone(
                    url,
                    CLONE_ROOT,
                    with_submodules=WITH_SUBMODULES,
                    with_lfs=WITH_LFS,
                    fetch_pr_refs=FETCH_PR_REFS,
                )
                rec["dir"] = str(d)
                rec["total_commits"] = get_total_commits(d)
                rec["status"] = "ok"
                ok += 1

                # --- New: GitHub PR / issue metadata collection ---
                if FETCH_GH_METADATA:
                    gh = parse_github_owner_repo(url)
                    if gh is not None:
                        owner, repo_name = gh
                        meta_dir = META_ROOT  # save all JSON files directly in metadataV1.0
                        try:
                            fetch_github_prs_and_issues(owner, repo_name, meta_dir)
                        except Exception as e:
                            msg = f"metadata error: {e}"
                            print(f"  [meta-error] {url}: {msg}")
                            # Optionally record the metadata error
                            if rec["status"] == "ok" and not rec["error"]:
                                rec["error"] = msg


            except subprocess.CalledProcessError as e:
                rec["status"] = "error"
                rec["error"] = (e.stderr or e.stdout or str(e)).strip()[:2000]
                fail += 1
            except Exception as e:
                rec["status"] = "error"
                rec["error"] = str(e)[:2000]
                fail += 1

            rec["seconds"] = round(time.time() - t0, 2)
            rows.append(rec)
            print(
                f"[{rec['status']}] {url} -> {rec['dir']} "
                f"({rec['seconds']}s)  commits={rec['total_commits']}"
            )

    # Write manifest
    MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["repo_url", "dir", "status", "seconds", "total_commits", "error"]
    with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


if __name__ == "__main__":
    main()


ℹ️ Loaded 6 GitHub token(s) from All_Tokens.env
Token lengths: [40, 40, 40, 40, 40, 40]
  [meta] Fetching PRs for connectbot/connectbot ...


In [2]:
import os
from dotenv import load_dotenv
import requests

load_dotenv(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

token = os.getenv("GITHUB_TOKEN_1")
print("Token length:", len(token) if token else None)

headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
resp = requests.get("https://api.github.com/user", headers=headers)
print(resp.status_code, resp.text[:200])


Token length: 40
200 {"login":"behnamparsa","id":66279475,"node_id":"MDQ6VXNlcjY2Mjc5NDc1","avatar_url":"https://avatars.githubusercontent.com/u/66279475?v=4","gravatar_id":"","url":"https://api.github.com/users/behnampar


## Step 2: Mining 
add a baseline to read the repos from the begining not from the first change

In [7]:
from __future__ import annotations

import csv
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor as PoolExecutor, as_completed
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

# ===========================
# Config (EDIT THESE PATHS)
# ===========================
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4")
INPUT_CLONES = WORK_ROOT / "clonesV1.0"
OUTPUT_MINE  = WORK_ROOT / "mineV1.0"
# Where PR / issues JSON files live (owner__repo_PRs.json, owner__repo_Issues.json)
META_ROOT    = WORK_ROOT / "metadataV1.0"



# Study end date (UTC)
CUTOFF_ISO = "2025-08-10 23:59:59 +0000"

# Timeline scope for evolution analysis:
#   "repo_wide" -> walk all origin/* heads (recommended for RQ3)
#   "default"   -> only default branch history
TIMELINE_SCOPE = "repo_wide"

# Performance / behavior toggles
MAX_REPOS = 0  # 0 = all
MAX_WORKERS = min(32, (os.cpu_count() or 8) * 2)
RESUME_IF_EXISTS = True
SUPPRESS_EMPTY_ROWS = True
CAPTURE_EMPTY_GAPS = True
EMIT_NONE_STATE = False

# Blob size guards
BLOB_MAX_SIZE = 2_000_000
HARD_MAX_SIZE = 5_000_000

# High-priority CI paths never skipped (even if large)
HIGH_PRIORITY_CI_PATHS = {
    ".github/workflows",
    ".gitlab-ci.yml",
    "azure-pipelines.yml",
    ".circleci/config.yml",
    ".bitrise.yml",
    ".travis.yml",
}

# ===========================
# Quick startup checks
# ===========================
def _failfast_checks() -> None:
    if shutil.which("git") is None:
        print("[fatal] Git not found in PATH. Install Git and/or add it to PATH.", file=sys.stderr)
        raise SystemExit(1)
    if not WORK_ROOT.exists():
        print(f"[fatal] WORK_ROOT does not exist: {WORK_ROOT}", file=sys.stderr)
        raise SystemExit(1)
    if not INPUT_CLONES.exists():
        print(f"[warn] INPUT_CLONES does not exist yet: {INPUT_CLONES}")
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

# ===========================
# File classifiers
# ===========================
YAML_EXTS = (".yml", ".yaml")
GRADLE_NAMES = {"build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts"}
GRADLE_EXTS = (".gradle", ".gradle.kts")
SCRIPT_EXTS = (".sh", ".bat", ".cmd", ".ps1")
CI_BUILD_SPECIAL = {"Jenkinsfile", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml"}
XML_BUILD_FILES = {"pom.xml", "build.xml", "config.xml"}

def is_yaml(path: str) -> bool:
    return os.path.splitext(path)[1].lower() in YAML_EXTS or os.path.basename(path) in CI_BUILD_SPECIAL

def is_gradle(path: str) -> bool:
    name = os.path.basename(path)
    ext = os.path.splitext(path)[1].lower()
    return name in GRADLE_NAMES or ext in GRADLE_EXTS

def is_script(path: str) -> bool:
    ext = os.path.splitext(path)[1].lower()
    return ext in SCRIPT_EXTS or os.path.basename(path) in ("Jenkinsfile",)

def is_ci_xml_or_build_xml(path: str) -> bool:
    return os.path.basename(path).lower() in {n.lower() for n in XML_BUILD_FILES}

def is_relevant_file(path: str) -> bool:
    return is_yaml(path) or is_gradle(path) or is_script(path) or is_ci_xml_or_build_xml(path)

def is_high_priority_ci_path(path: str) -> bool:
    norm = path.replace("\\", "/")
    base = os.path.basename(norm)
    if base in HIGH_PRIORITY_CI_PATHS:
        return True
    for root in HIGH_PRIORITY_CI_PATHS:
        if norm.startswith(root.rstrip("/") + "/"):
            return True
    return False

# ===========================
# Regex helpers & normalizers
# ===========================
COMMENT_LINE_RE = re.compile(r"(?m)^\s*(#|//|REM\b|::).*?$")

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# --- Gradle comment stripper (preserves http(s)://) ---
def strip_comments_gradle(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.S)                 # /* ... */
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.M)                  # // ...
    return s

def normalize_block_keys(text: str) -> str:
    # expose content of run/script/command keys and drop YAML dashes before likely shell lines
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$", r"\2", text)
    text = re.sub(r"(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$", "", text)
    text = re.sub(
        r"(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)",
        r"\1",
        text,
    )
    return text

IGNORE_GHA_ACTIONS_RE = re.compile(
    r"(?mi)^\s*uses\s*:\s*(docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)|actions/checkout|docker/setup-qemu-action|docker/setup-buildx-action)@.*$"
)

def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub("", text or "")

GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def pre_sanitize(text: str) -> str:
    return GHA_EXPR_RE.sub("", text or "")

# Gradle command shapes (for CI scripts/YAML)
GRADLE_PREFIX = (
    r"^\s*"
    r"(?:\S+=\S+\s+)*"
    r"(?:sudo\s+)?"
    r"(?:(?:bash|sh)\s+-c[l]?\s+[\'\"]?)?"
    r"(?:[^#\n;]*?&&\s+)?"
    r"(?:cd\s+\S+\s+&&\s+)?"
    r"(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?"
)
GRADLE_ANYWHERE = r"(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*"
GRADLE_ANYWHERE_RE = re.compile(GRADLE_ANYWHERE)
GRADLE_BUILD_ACTION_RE = re.compile(r"(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@")
NON_TEST_PREFIX = r"(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)"
SHELL_PREFIX = r"(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'\"]?)?(?:[^#\n;]*?&&\s+)?"

# Device / env patterns (CI side)
EMULATOR_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+"
ADB_WAIT_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b"
ADB_SERIAL_LINE = rf"(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)"
REAL_DEVICE_LINE = rf"(?mi)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b"

def compile_any(patterns: List[str], flags: int = re.I | re.M) -> List[re.Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable, text: str) -> bool:
    for p in patterns:
        if isinstance(p, str):
            p = re.compile(p, re.I | re.M)
        if p.search(text):
            return True
    return False

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# -------- DEVICE SOURCES --------
DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [REAL_DEVICE_LINE]),
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device", [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@", [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r"(?mi)^\s*(?:\./)?android-wait-for-emulator\b"]),
    ("Emulator", "start-emulator.sh", [r"(?mi)^\s*start-emulator\.sh\b"]),
    ("Emulator", "android create avd", [r"\bandroid\b[^\n]*\bcreate\s+avd\b"]),
    # ("Emulator", "circleci android orb", [
    #     r"(?mi)^\s*(?:-\s*)?android/start-emulator-and-run-tests\s*:",
    #     r"(?mi)^\s*system-image\s*:\s*system-images;android-\d+;google_apis;"
    # ]),
    ("Emulator", "circleci android orb", [
    r"(?mi)^\s*(?:-\s*)?android/(?:start-emulator-and-run-tests|create-avd|launch-emulator)\s*:",
    r"(?mi)^\s*system-image\s*:\s*system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*);(?:default|google_apis)[^\s]*"
    ]),
    ("Emulator", "reactivecircus runner", [r"(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+"]),
    ("Emulator", "malinskiy runner", [r"(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+"]),
    ("Emulator", "sys-img component", [
        r"(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b",
        r"(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "avdmanager", [r"(?m)^\s*\S*avdmanager\b"]),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r"^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b"
    ]),
    ("Emulator", "other gha emulator", [
        r"(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+",
        r"(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)(?!malinskiy/action-android/emulator-run-cmd@)(?!emulator-wtf/run-tests@)[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+"
    ]),
    ("Third_Party_Lab", "gcloud firebase", [r"(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "browserstack/bstack", [r"(?i)\b(browserstack|bstack)\b"]),
    ("Third_Party_Lab", "appcenter test", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "maestro cloud", [r"(?mi)^[^\n]*\bmaestro\s+cloud\b"]),
    ("Third_Party_Lab", "emulator.wtf action", [
        r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+",
        r"(?i)\bemulator\.wtf\b"
    ]),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

# -------- Triggers (CI/YAML side) --------
TRIGGER_SOURCES_PRIMARY = [
    ("Gradle", "connectedAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*"]),
    ("Gradle", "connected.*Android.*", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b(?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)[^\n\r]*"]),
    ("Gradle", "connectedCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*"]),
    ("Gradle", "cAT shorthand", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*"]),
    ("Gradle", "deviceCheck", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*"]),
    ("Gradle", "managedDevice AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "variant/device AndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b[^\n\r]*"]),
    ("Gradle", "Spoon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b"]),
    ("ADB", "am instrument", [r"(?mi)^[^\n]*\bam\s+instrument\b"]),
    ("Third_Party_Lab", "gcloud firebase (instr)", [r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)"]),
    ("Third_Party_Lab", "flank", [r"(?mi)^[^\n]*\bflank\s+android\s+run\b"]),
    ("Third_Party_Lab", "saucectl", [r"(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b"]),
    ("Third_Party_Lab", "appcenter", [r"(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b"]),
    ("Third_Party_Lab", "emulator.wtf run", [r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+", r"(?i)\bemulator\.wtf\b"]),
]
TRIGGER_SOURCES_PRIMARY += [
    ("Gradle", "generateBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "collectBaselineProfile", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*"]),
    ("Gradle", "connectedBenchmarkAndroidTest", [rf"(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*"]),
]

TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b"]),
    ("Gradle", "connectedAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b"]),
    ("Gradle", "connectedCheck (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connectedcheck\b"]),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b"]),
    ("Gradle", "variant/device AndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\b(?:(?:[:\w-]+:)*) (?!{NON_TEST_PREFIX})[\w-]*androidtest\b"]),
    ("Gradle", "Spoon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b"]),
    ("Gradle", "Marathon (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b"]),
]
TRIGGER_SOURCES_ANYWHERE += [
    ("Gradle", "generateBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bgenerate(?:\w*?)baselineprofile\b"]),
    ("Gradle", "collectBaselineProfile (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bcollect(?:\w*?)baselineprofile\b"]),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)", [rf"(?mi){GRADLE_ANYWHERE}\bconnectedbenchmarkandroidtest\b"]),
]

TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

def collect_hits_with_groups(patterns, text: str):
    labels = []
    groups = []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# GHA gradle inputs guard
GHA_GRADLE_INPUTS = compile_any([
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:(?:[:\w-]+:)*) (?!assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)[\w-]*androidtest\b",
    r"(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b",
])

PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@", r"(?i)\bemulator\.wtf\b"])),
    ("firebase-test-lab", compile_any([r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b", r"(?mi)\bflank\s+android\s+run\b"])),
    ("browserstack", compile_any([r"(?i)\bbrowserstack\b", r"(?i)\bbstack\b"])),
    ("sauce-labs", compile_any([r"(?mi)\bsaucectl(?:\s+run)?\b", r"(?mi)\bsauce\s+ctl\b"])),
    ("appcenter", compile_any([r"(?mi)\bappcenter\s+test\s+run\s+android\b"])),
    ("maestro-cloud", compile_any([r"(?mi)\bmaestro\s+cloud\b"])),
]

INLINE_DEVICE_HINTS = compile_any([
    r"(?mi)^\s*devices\s*:\s*\|",
    r"(?mi)\b--device\b",
    r"(?mi)\bmodel\s*=\s*[^,\s]+",
    r"(?mi)\bversion\s*=\s*\d+",
    r"(?mi)\blocale\s*=\s*[-\w]+",
    r"(?mi)\borientation\s*=\s*(portrait|landscape)",
    r"(?mi)^\s*with-orchestrator\s*:\s*true\b",
    r"(?mi)\b--use-orchestrator\b",
    r"(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b",
    r"(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b",
])

CONFIG_FILE_HINTS = compile_any([
    r"(?mi)\.ewtf\.ya?ml\b",
    r"(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b",
    r"(?mi)\b--config(?:=|\s+)\S+",
    r"(?mi)\b(browserstack\.ya?ml)\b",
    r"(?mi)\b(bs(?:config)?\.ya?ml)\b",
])

ANDROID_CONTEXT_RE = re.compile(
    r"(?i)\b(adb|avd|emulator|android\s+sdk|system-images;android-|androidtest|connected(?:check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run)\b"
)

EMULATOR_COMMUNITY_LABELS = {"reactivecircus runner", "malinskiy runner", "other gha emulator", "circleci android orb"}
EMULATOR_CUSTOM_LABELS = {
    "adb -s emulator-serial",
    "adb wait-for-device",
    "emulator -avd/@",
    "android-wait-for-emulator",
    "start-emulator.sh",
    "android create avd",
    "avdmanager",
    "sdkmanager system-images/emulator",
    "sys-img component",
}
THIRD_PARTY_STRONG_LABELS = {"gcloud firebase", "emulator.wtf run", "saucectl", "appcenter", "browserstack/bstack", "maestro cloud"}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
STRONG_DEVICE_LABELS = EMULATOR_COMMUNITY_LABELS | EMULATOR_CUSTOM_LABELS | THIRD_PARTY_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def has_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def has_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

# ===========================
# Gradle GMD config detector (aligned with V6.0)
# ===========================
TESTOPTIONS_HEAD = re.compile(r"\btestOptions\s*\{", re.I)
MANAGED_HEAD     = re.compile(r"\bmanagedDevices\s*\{", re.I)
GMD_INNER_TOKENS_RX = re.compile(r"\b(managedDevices|managedVirtualDevice|devices|groups|deviceGroups)\b", re.I)

def _find_block_span_from_head(text: str, head_start: int) -> Optional[Tuple[int, int]]:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

def find_gmd_block(text_gradle_no_comments: str) -> bool:
    """Return True if we see testOptions{ managedDevices{ … } } or a direct managedDevices{ … } block."""
    t = text_gradle_no_comments
    # Direct managedDevices { ... }
    for m in MANAGED_HEAD.finditer(t):
        if _find_block_span_from_head(t, m.start()):
            return True
    # testOptions { ... } containing GMD tokens
    for m in TESTOPTIONS_HEAD.finditer(t):
        ci = _find_block_span_from_head(t, m.start())
        if not ci:
            continue
        a, b = ci
        sub = t[a:b+1]
        if GMD_INNER_TOKENS_RX.search(sub):
            return True
    return False

# ===========================
# Git helpers
# ===========================
def run_git(repo: Path, args: List[str], text: bool = True, check: bool = True, retries: int = 2) -> subprocess.CompletedProcess:
    last_exc: Optional[BaseException] = None
    for attempt in range(retries + 1):
        try:
            return subprocess.run(
                ["git", "-C", str(repo)] + args,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=text,
                check=check,
                encoding="utf-8" if text else None,
                errors="replace" if text else None,
            )
        except subprocess.CalledProcessError as e:
            last_exc = e
            time.sleep(0.1 * (attempt + 1) + random.random() * 0.1)
        except Exception as e:
            last_exc = e
            time.sleep(0.05)
    if isinstance(last_exc, subprocess.CalledProcessError) and not check:
        return last_exc  # type: ignore[return-value]
    raise last_exc  # type: ignore[misc]

def list_repos(root: Path) -> List[Path]:
    out: List[Path] = []
    for p, dirs, _files in os.walk(root):
        pth = Path(p)
        if (pth / ".git").exists():
            out.append(pth)
            dirs[:] = []
    return out

def default_branch_ref(repo: Path) -> Optional[str]:
    try:
        ref = run_git(repo, ["symbolic-ref", "refs/remotes/origin/HEAD"]).stdout.strip()
        if ref.startswith("refs/remotes/"):
            parts = ref.split("/")
            return "/".join(parts[2:])
    except Exception:
        pass
    for cand in ("origin/main", "origin/master"):
        cp = run_git(repo, ["rev-parse", "--verify", cand], check=False)
        if cp.returncode == 0:
            return cand
    return None

def cutoff_head_default(repo: Path, cutoff_iso: str) -> List[str]:
    ref = default_branch_ref(repo)
    if not ref:
        return []
    cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", ref], check=False)
    sha = (cp.stdout or "").strip()
    return [sha] if sha else []

def cutoff_heads_all(repo: Path, cutoff_iso: str) -> List[str]:
    refs_raw = run_git(repo, ["for-each-ref", "--format=%(refname:short)", "refs/remotes/origin"]).stdout
    refs = [r for r in refs_raw.splitlines() if r and r != "origin/HEAD" and not r.startswith("origin/pr/")]
    shas: List[str] = []
    for r in refs:
        cp = run_git(repo, ["rev-list", "-n1", "--first-parent", f"--before={cutoff_iso}", r], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    if not shas:
        cp = run_git(repo, ["rev-list", "-n1", "--before", cutoff_iso, "--all"], check=False)
        sha = (cp.stdout or "").strip()
        if sha:
            shas.append(sha)
    return shas

def _pathspecs_yaml() -> List[str]:
    return [":(glob)**/*.yml", ":(glob)**/*.yaml", ".travis.yml", "azure-pipelines.yml", ".gitlab-ci.yml", "circle.yml", ".bitrise.yml"]

def _pathspecs_gradle() -> List[str]:
    return ["build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts", ":(glob)**/*.gradle", ":(glob)**/*.gradle.kts"]

def _pathspecs_scripts_and_xml() -> List[str]:
    return ["Jenkinsfile", ":(glob)**/*.sh", ":(glob)**/*.bat", ":(glob)**/*.cmd", ":(glob)**/*.ps1", ":(glob)**/pom.xml", ":(glob)**/build.xml", ":(glob)**/config.xml"]

def commits_touching_relevant(repo: Path, heads: List[str]) -> List[str]:
    pathspecs = _pathspecs_yaml() + _pathspecs_gradle() + _pathspecs_scripts_and_xml()
    if not heads:
        return []
    s = run_git(repo, ["rev-list", "--reverse"] + heads + ["--"] + pathspecs).stdout
    return [c for c in s.splitlines() if c.strip()]

def list_tree_entries(repo: Path, treeish: str) -> List[Tuple[str, str]]:
    try:
        raw = run_git(repo, ["ls-tree", "-r", "-z", treeish]).stdout
    except Exception:
        raw = ""
    out: List[Tuple[str, str]] = []
    for entry in raw.split("\x00"):
        if not entry or "\t" not in entry:
            continue
        meta, path = entry.split("\t", 1)
        parts = meta.split()
        if len(parts) < 3:
            continue
        sha = parts[2]
        out.append((sha, path))
    return out

def blob_size(repo: Path, sha: str) -> int:
    try:
        return int(run_git(repo, ["cat-file", "-s", sha]).stdout.strip())
    except Exception:
        return 0

@lru_cache(maxsize=250_000)
def read_blob_cached(repo_path: str, sha: str) -> Optional[str]:
    repo = Path(repo_path)
    try:
        return run_git(repo, ["cat-file", "-p", sha]).stdout
    except Exception:
        return None

# ===========================
# PR / Issues metadata helpers
# ===========================

def _split_owner_repo_from_folder(folder_name: str) -> Tuple[Optional[str], str]:
    """
    Our clone dirs are owner__repo. Return (owner, repo).
    If it doesn't match that pattern, owner is None and repo is folder_name.
    """
    if "__" in folder_name:
        owner, repo = folder_name.split("__", 1)
        return owner, repo
    return None, folder_name


def _load_json_if_exists(path: Path):
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _load_repo_metadata(repo: Path) -> Tuple[List[dict], List[dict]]:
    """
    Given a cloned repo path, load PRs and Issues JSON from META_ROOT using
    the owner__repo_PRs.json and owner__repo_Issues.json naming convention.
    Returns (prs, issues) lists. If files are missing, returns empty lists.
    """
    owner, short_repo = _split_owner_repo_from_folder(repo.name)
    base_name = f"{owner}__{short_repo}" if owner else short_repo

    prs_path    = META_ROOT / f"{base_name}_PRs.json"
    issues_path = META_ROOT / f"{base_name}_Issues.json"

    prs    = _load_json_if_exists(prs_path) or []
    issues = _load_json_if_exists(issues_path) or []

    # Ensure they are lists (GitHub API returns list for those endpoints)
    if not isinstance(prs, list):
        prs = []
    if not isinstance(issues, list):
        issues = []

    return prs, issues


def _get_commit_subject_body(repo: Path, commit: str,
                             _cache: Dict[str, Tuple[str, str]]) -> Tuple[str, str]:
    """
    Return (subject, body) for a commit, with a small per-repo cache.
    """
    if commit in _cache:
        return _cache[commit]

    try:
        cp = run_git(
            repo,
            ["show", "-s", "--format=%s%n%b", commit],
            text=True,
            check=True,
        )
        out = cp.stdout or ""
    except Exception:
        _cache[commit] = ("", "")
        return ("", "")

    lines = out.splitlines()
    subject = lines[0] if lines else ""
    body = "\n".join(lines[1:]).strip()
    _cache[commit] = (subject, body)
    return subject, body


_CLOSING_ISSUE_RX = re.compile(
    r"(?i)\b(?:close[sd]?|fixe?[sd]?|resolve[sd]?)\s+#(\d+)\b"
)


def _build_env_event_rows_for_repo(repo: Path, rec: Dict) -> List[Dict]:
    """
    For a given repo and its timeline JSON record (rec),
    build a flat list of rows for env-related events,
    enriched with commit + PR + Issues data when available.
    """
    rows: List[Dict] = []

    prs, issues = _load_repo_metadata(repo)

    # Build quick lookup tables ------------------------------------
    pr_by_merge_sha: Dict[str, dict] = {}
    pr_by_head_sha: Dict[str, dict] = {}
    pr_to_issue_nums: Dict[int, List[int]] = {}

    issues_by_number: Dict[int, dict] = {}
    for iss in issues:
        try:
            num = iss.get("number")
            if isinstance(num, int):
                issues_by_number[num] = iss
        except Exception:
            continue

    for pr in prs:
        # map merge_commit_sha -> PR
        msha = pr.get("merge_commit_sha")
        if msha:
            pr_by_merge_sha[msha] = pr

        # map head.sha -> PR (useful for fast-forward merges or if merge_commit_sha missing)
        head = pr.get("head")
        if isinstance(head, dict):
            hsha = head.get("sha")
            if hsha:
                pr_by_head_sha[hsha] = pr

        # find 'closes #123' style references in PR body
        body = pr.get("body") or ""
        try:
            nums = {int(n) for n in _CLOSING_ISSUE_RX.findall(body) if n.isdigit()}
        except Exception:
            nums = set()
        if nums and isinstance(pr.get("number"), int):
            pr_to_issue_nums[int(pr["number"])] = sorted(nums)

    owner, short_repo = _split_owner_repo_from_folder(repo.name)
    full_name = f"{owner}/{short_repo}" if owner else short_repo

    # We'll reuse commit text across rows
    commit_cache: Dict[str, Tuple[str, str]] = {}

    events = rec.get("events", {}) or {}

    for style, elist in events.items():
        for e in elist:
            commit = e.get("commit")
            if not commit:
                continue
            date        = e.get("date") or ""
            change_type = e.get("event") or ""
            on_default  = e.get("on_default", 0)

            subject, body = _get_commit_subject_body(repo, commit, commit_cache)

            # Try to associate this commit with a PR
            pr = pr_by_merge_sha.get(commit) or pr_by_head_sha.get(commit)

            pr_number = ""
            pr_state = ""
            pr_title = ""
            pr_body = ""
            pr_created_at = ""
            pr_merged_at = ""
            pr_closed_at = ""
            pr_user_login = ""
            pr_labels = ""
            pr_html_url = ""
            issues_summary = ""

            if pr:
                pr_number = str(pr.get("number", ""))
                pr_state = pr.get("state", "") or ""
                pr_title = (pr.get("title") or "").replace("\n", " ").strip()
                pr_body = (pr.get("body") or "").strip()
                pr_created_at = pr.get("created_at", "") or ""
                pr_merged_at = pr.get("merged_at", "") or ""
                pr_closed_at = pr.get("closed_at", "") or ""
                user = pr.get("user") or {}
                if isinstance(user, dict):
                    pr_user_login = user.get("login", "") or ""
                labels = pr.get("labels") or []
                if isinstance(labels, list):
                    label_names = []
                    for lab in labels:
                        if isinstance(lab, dict) and "name" in lab:
                            label_names.append(str(lab["name"]))
                    pr_labels = ";".join(label_names)
                pr_html_url = pr.get("html_url", "") or ""

                # Attach closing issues summary, if any
                try:
                    n_int = int(pr.get("number"))
                except Exception:
                    n_int = None

                if n_int is not None:
                    nums = pr_to_issue_nums.get(n_int, [])
                    issue_texts: List[str] = []
                    for num in nums:
                        iss = issues_by_number.get(num)
                        if not iss:
                            continue
                        ttl = (iss.get("title") or "").replace("\n", " ").strip()
                        st = iss.get("state", "") or ""
                        issue_texts.append(f"#{num} [{st}] {ttl}")
                    if issue_texts:
                        issues_summary = " || ".join(issue_texts)

            rows.append(
                {
                    "full_name": full_name,            # owner/repo
                    "repo_folder": repo.name,          # owner__repo
                    "env_style": style,                # Emu_Community, GMD, ThirdParty...
                    "change_type": change_type,        # added / removed / maintenance
                    "change_date": date,
                    "on_default": on_default,

                    "commit": commit,
                    "commit_short": commit[:12],
                    "commit_subject": subject,
                    "commit_body": body,

                    "pr_number": pr_number,
                    "pr_state": pr_state,
                    "pr_title": pr_title,
                    "pr_body": pr_body,
                    "pr_created_at": pr_created_at,
                    "pr_merged_at": pr_merged_at,
                    "pr_closed_at": pr_closed_at,
                    "pr_user_login": pr_user_login,
                    "pr_labels": pr_labels,
                    "pr_html_url": pr_html_url,

                    "issues_summary": issues_summary,
                }
            )

    return rows



# ===========================
# Detection at snapshot level
# ===========================
def scan_snapshot_for_styles(repo: Path, treeish: str) -> Tuple[Set[str], Dict]:
    entries = list_tree_entries(repo, treeish)

    # Aggregate features
    features = {
        "yaml_loc": 0,
        "gradle_loc": 0,
        "runs_on": set(),
        "matrix_width_hint": 0,
        "api_levels": set(),
        "reliability_flags": set(),
        "gmd_config_present": False,
        "gmd_identifiers": set(),  # optional, we only store names when we parse them from block heads
    }
    all_text_blocks: List[Tuple[str, str]] = []
    raw_blocks_for_guard: List[str] = []
    has_gradle_anywhere_repo = False

    RX_RUNS_ON = re.compile(r"(?mi)^\s*runs-on\s*:\s*(.+)$")
    RX_LIST_ITEM = re.compile(r"(?mi)^\s*-\s")
    RX_API_LEVEL = re.compile(r"(?i)\bandroid[-_ ]?(\d{2})\b|system-images;android-(\d{2})\b")
    RX_RETRY = re.compile(r"(?i)\bretry\b|\bmax-attempts\b|\bflaky\b")
    RX_TIMEOUT = re.compile(r"(?i)\btimeout[- ]?minutes\b|\btimeout\b")
    RX_HEADLESS = re.compile(r"(?i)\bheadless\b|\b-no-window\b|\b-no-boot-anim\b")
    RX_WAIT_FOR_DEVICE = re.compile(r"(?i)\badb\s+wait[- ]?for[- ]?device\b")

    # Optional identifier capture in GMD blocks (names before "(ManagedVirtualDevice)")
    GMD_BLOCK_NAME_RX = re.compile(
        r'(\w+)\s*\(\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
        re.I
    )

    for sha, path in entries:
        if not is_relevant_file(path):
            continue
        size = blob_size(repo, sha)
        if not is_high_priority_ci_path(path) and BLOB_MAX_SIZE and size > BLOB_MAX_SIZE:
            continue
        text = read_blob_cached(str(repo), sha)
        if not text or len(text) > HARD_MAX_SIZE:
            continue

        lines = text.splitlines()
        norm = text  # default

        if is_yaml(path):
            features["yaml_loc"] += len(lines)
            for m in RX_RUNS_ON.finditer(text):
                features["runs_on"].add(m.group(1).strip())
            features["matrix_width_hint"] = max(features["matrix_width_hint"], len(RX_LIST_ITEM.findall(text)))
            # sanitize
            norm = strip_comments(norm)
            norm = normalize_block_keys(norm)
            norm = strip_irrelevant_ci_lines(norm)
            norm = pre_sanitize(norm)

        elif is_script(path) or is_ci_xml_or_build_xml(path):
            norm = strip_comments(norm)
            norm = normalize_block_keys(norm)
            norm = strip_irrelevant_ci_lines(norm)
            norm = pre_sanitize(norm)

        elif is_gradle(path):
            features["gradle_loc"] += len(lines)
            # For Gradle, use Gradle-aware comment stripping then look for GMD config blocks
            gradle_clean = strip_comments_gradle(text)
            # Detect managedDevices presence (aligned with your V6.0)
            if not features["gmd_config_present"] and find_gmd_block(gradle_clean):
                features["gmd_config_present"] = True
                # best-effort: capture identifiers (optional, for diagnostics only)
                for m in GMD_BLOCK_NAME_RX.finditer(gradle_clean):
                    ident = (m.group(1) or "").strip()
                    if ident:
                        features["gmd_identifiers"].add(ident)
            # Keep norm as lowercased “clean” text for any incidental signals
            norm = gradle_clean

        # Common feature mining from the (possibly sanitized) text
        for m in RX_API_LEVEL.finditer(text):
            for g in m.groups():
                if g and g.isdigit():
                    features["api_levels"].add(int(g))
        if RX_RETRY.search(text):
            features["reliability_flags"].add("retry")
        if RX_TIMEOUT.search(text):
            features["reliability_flags"].add("timeout")
        if RX_HEADLESS.search(text):
            features["reliability_flags"].add("headless")
        if RX_WAIT_FOR_DEVICE.search(text):
            features["reliability_flags"].add("wait_for_device")

        all_text_blocks.append((path, norm))
        raw_blocks_for_guard.append(norm)

        if GRADLE_ANYWHERE_RE.search(norm) or GRADLE_BUILD_ACTION_RE.search(norm):
            has_gradle_anywhere_repo = True

    # Collect triggers & devices repo-wide (mostly from YAML/scripts)
    trigger_labels: List[str] = []
    trigger_groups: List[str] = []
    device_labels: List[str] = []
    device_groups: List[str] = []

    for _path, norm in all_text_blocks:
        nlow = norm.lower()

        t_labels, t_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, nlow)
        fb_labels, fb_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, nlow)
        if fb_labels:
            t_labels = unique_preserve(t_labels + fb_labels)
            t_groups = unique_preserve(t_groups + fb_groups)

        # GHA gradle input guard
        if any_match(GHA_GRADLE_INPUTS, nlow):
            gha_tied = re.search(
                r"(?mi)^\s*uses\s*:\s*(reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd)@",
                nlow,
            )
            if has_gradle_anywhere_repo or gha_tied:
                if "gha gradle inputs/script" not in t_labels:
                    t_labels.append("gha gradle inputs/script")
                if "Gradle" not in t_groups:
                    t_groups.append("Gradle")

        trigger_labels = unique_preserve(trigger_labels + t_labels)
        trigger_groups = unique_preserve(trigger_groups + t_groups)

        d_labels, d_groups = collect_hits_with_groups(DEVICE_PATTERNS, nlow)
        # Guard: “other gha emulator” must have Android context or a trigger
        if ("other gha emulator" in d_labels) and (not ANDROID_CONTEXT_RE.search(nlow)) and (not t_labels):
            d_labels = [l for l in d_labels if l != "other gha emulator"]
            if not d_labels:
                d_groups = [g for g in d_groups if g != "Emulator"]

        device_labels = unique_preserve(device_labels + d_labels)
        device_groups = unique_preserve(device_groups + d_groups)

    has_test_trigger = bool(trigger_labels)

    # Weak hint suppression: require strong device OR a test trigger
    strong_seen = bool(set(device_labels) & STRONG_DEVICE_LABELS)
    if not (strong_seen or has_test_trigger):
        device_labels = [l for l in device_labels if l in STRONG_DEVICE_LABELS]
        if not device_labels:
            device_groups = []

    # BrowserStack upload-only guard
    repo_text = "\n".join(raw_blocks_for_guard)
    provider = detect_provider(repo_text)
    inline_decl = has_inline_env(repo_text)
    config_decl = has_config_env(repo_text)
    has_ftl_instr = bool(re.search(r"(?mi)\bgcloud\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)", repo_text))
    bs_test_hint = re.search(r"(?mi)\b(browserstack|bstack)\b[^\n]*\b(espresso|instrumentation|app-automate|automate|--device|--devices)\b", repo_text)
    if ("browserstack/bstack" in device_labels or provider == "browserstack"):
        if (not has_test_trigger) and (not inline_decl) and (not config_decl) and (not has_ftl_instr) and (not bs_test_hint):
            device_labels = [l for l in device_labels if l != "browserstack/bstack"]
            if "Third_Party_Lab" in device_groups and not any(l in {"gcloud firebase", "saucectl", "appcenter", "maestro cloud", "emulator.wtf action"} for l in device_labels):
                device_groups = [g for g in device_groups if g != "Third_Party_Lab"]

    # Map to final styles
    styles: Set[str] = set()

    if "Third_Party_Lab" in device_groups:
        styles.add("ThirdParty")

    if "Emulator" in device_groups:
        lbls = set(device_labels)
        if lbls & EMULATOR_COMMUNITY_LABELS:
            styles.add("Emu_Community")
        elif lbls & EMULATOR_CUSTOM_LABELS:
            styles.add("Emu_Custom")

    # --- GMD detection parity with shallow snapshot ---
    # Count GMD if EITHER:
    #  (A) GMD Gradle configuration is present, OR
    #  (B) CI shows managed-device AndroidTest triggers (execution evidence).
    def has_gmd_gradle_trigger(tlabs: List[str]) -> bool:
        L = {l.lower() for l in tlabs}
        if any("manageddevice androidtest" in l for l in L):
            return True
        if ("variant/device androidtest" in L) and not any(x in L for x in {"connected.*android.*", "connectedandroidtest", "connectedbenchmarkandroidtest", "spoon", "marathon"}):
            return True
        return False

    if features["gmd_config_present"] or has_gmd_gradle_trigger(trigger_labels):
        styles.add("GMD")

    meta = {
        "sources": {"yaml": [], "gradle": [], "scripts_xml": []},
        "features": {
            "yaml_loc": features["yaml_loc"],
            "gradle_loc": features["gradle_loc"],
            "runs_on": sorted(features["runs_on"]),
            "matrix_width_hint": features["matrix_width_hint"],
            "api_levels": sorted(int(x) for x in features["api_levels"]),
            "reliability_flags": sorted(features["reliability_flags"]),
            "trigger_labels": sorted(trigger_labels),
            "device_labels": sorted(device_labels),
            "provider": provider,
            "inline_env": bool(inline_decl),
            "config_env": bool(config_decl),
            "gradle_present_repo": bool(has_gradle_anywhere_repo),
            "gmd_config_present": bool(features["gmd_config_present"]),
            "gmd_identifiers": sorted(features["gmd_identifiers"]) if features["gmd_identifiers"] else [],
        },
    }
    return styles, meta

# ===========================
# Timeline building
# ===========================
def empty_result(repo: Path, cutoff_iso: str) -> Dict:
    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_default_head": [],
        "cutoff_all_heads": [],
        "first_commit_date": None,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": [],
        "events": {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []},
        "snapshot_as_of_cutoff_default": [],
        "snapshot_as_of_cutoff_repo_wide": [],
        "off_default_extra_styles": [],
        "has_off_default_extra": 0,
        "empty_gaps": [],
        "qa_issue": "No environment detected at cutoff",
    }

def earliest_commit(repo: Path) -> Optional[str]:
    try:
        s = run_git(repo, ["rev-list", "--max-parents=0", "--all"]).stdout.strip()
        return s.splitlines()[0] if s else None
    except Exception:
        return None

def get_commit_date_iso(repo: Path, commit: str) -> Optional[str]:
    try:
        return run_git(repo, ["show", "-s", "--format=%cI", commit]).stdout.strip() or None
    except Exception:
        return None

def union_snapshot_styles_at_cutoff(repo: Path, heads: List[str]) -> List[str]:
    styles: Set[str] = set()
    for sha in heads:
        s, _m = scan_snapshot_for_styles(repo, sha)
        styles |= s
    return sorted(styles)

def build_timeline(repo: Path, cutoff_iso: str) -> Dict:
    # Heads at cutoff
    default_heads = cutoff_head_default(repo, cutoff_iso)
    all_heads     = cutoff_heads_all(repo, cutoff_iso)

    if not default_heads and not all_heads:
        return empty_result(repo, cutoff_iso)

    # Commits touching relevant files on default and repo-wide (for tagging & scope)
    commits_default = commits_touching_relevant(repo, default_heads) if default_heads else []
    commits_all     = commits_touching_relevant(repo, all_heads) if all_heads else commits_default

    # Date maps
    commit_dates: Dict[str, str] = {}
    for c in commits_all:
        d = get_commit_date_iso(repo, c)
        if d:
            commit_dates[c] = d
    first_commit_date = commit_dates.get(commits_all[0]) if commits_all else None

    # Build timeline (scope-controlled)
    scope_commits = commits_all if TIMELINE_SCOPE == "repo_wide" else commits_default
    scope_commits = scope_commits or commits_all  # fallback

    timeline: List[Dict] = []
    events: Dict[str, List[Dict]] = {"Emu_Community": [], "Emu_Custom": [], "GMD": [], "ThirdParty": []}
    prev: Set[str] = set()

    empty_gaps: List[Dict] = []
    gap_open: Optional[Dict] = None

    # (Optional) historical baseline at repository root
    root = earliest_commit(repo)
    if root:
        base_styles, base_meta = scan_snapshot_for_styles(repo, root)
        if base_styles:
            baseline_date = get_commit_date_iso(repo, root)
            timeline.append({"date": baseline_date, "commit": root, "on_default": int(root in commits_default), "styles": sorted(base_styles), **base_meta})
            for s in sorted(base_styles):
                if s in events:
                    events[s].append({"event": "added", "date": baseline_date, "commit": root, "on_default": int(root in commits_default)})
            prev = set(base_styles)

    default_set = set(commits_default)

    for c in scope_commits:
        styles_now, meta_now = scan_snapshot_for_styles(repo, c)
        dt = commit_dates.get(c)
        on_default = int(c in default_set)

        # Skip commits that didn't detect any relevant style signals (unless capturing 'empty' gaps)
        if SUPPRESS_EMPTY_ROWS and not styles_now:
            if CAPTURE_EMPTY_GAPS and gap_open is None:
                gap_open = {"start_date": dt, "start_commit": c, "prev_state": "+".join(sorted(prev)) if prev else ""}
            continue

        # If we were in a gap and now have styles, close the gap (existing behavior)
        if CAPTURE_EMPTY_GAPS and gap_open is not None and styles_now:
            gap_open["end_date"] = dt
            gap_open["end_commit"] = c
            gap_open["next_state"] = "+".join(sorted(styles_now))
            try:
                d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
                d2 = datetime.fromisoformat((dt or "").replace("Z", "+00:00")).replace(tzinfo=None)
                gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
            except Exception:
                gap_open["duration_days"] = None
            gap_open["censored"] = False
            empty_gaps.append(gap_open)
            gap_open = None

        if styles_now != prev:
            # === boundary change: keep existing behavior ===
            if EMIT_NONE_STATE and not styles_now:
                timeline.append({"date": dt, "commit": c, "on_default": on_default, "styles": [], **meta_now})
            elif styles_now:
                timeline.append({"date": dt, "commit": c, "on_default": on_default, "styles": sorted(styles_now), **meta_now})

            added = styles_now - prev
            removed = prev - styles_now
            for s in sorted(added):
                if s in events:
                    events[s].append({"event": "added", "date": dt, "commit": c, "on_default": on_default})
            for s in sorted(removed):
                if s in events:
                    events[s].append({"event": "removed", "date": dt, "commit": c, "on_default": on_default})
            prev = styles_now

        else:
            # === NEW: within-episode maintenance (style set unchanged) ===
            # We reached this commit because it touched relevant paths already (by construction).
            # Treat it as "maintenance" for all styles currently active.
            if styles_now:
                for s in sorted(styles_now):
                    if s in events:
                        events[s].append({
                            "event": "maintenance",
                            "date": dt,
                            "commit": c,
                            "on_default": on_default
                        })
            # No timeline row needed here; we only record the maintenance touch.

    # Cutoff snapshots (both lenses)
    snapshot_default   = union_snapshot_styles_at_cutoff(repo, default_heads) if default_heads else []
    snapshot_repo_wide = union_snapshot_styles_at_cutoff(repo, all_heads) if all_heads else snapshot_default

    if CAPTURE_EMPTY_GAPS and gap_open is not None:
        # close last gap at cutoff
        gap_open["end_date"] = cutoff_iso
        gap_open["end_commit"] = (default_heads or all_heads or [None])[0]
        gap_open["next_state"] = ""
        try:
            d1 = datetime.fromisoformat((gap_open["start_date"] or "").replace("Z", "+00:00")).replace(tzinfo=None)  # type: ignore[index]
            d2 = datetime.fromisoformat(cutoff_iso.replace("Z", "+00:00")).replace(tzinfo=None)
            gap_open["duration_days"] = round(max(0.0, (d2 - d1).total_seconds() / 86400.0), 2)
        except Exception:
            gap_open["duration_days"] = None
        gap_open["censored"] = True
        empty_gaps.append(gap_open)
        gap_open = None

    off_default_extra = sorted(set(snapshot_repo_wide) - set(snapshot_default))
    qa_issue = None
    if not snapshot_default and not snapshot_repo_wide:
        qa_issue = "No environment detected at cutoff"

    return {
        "repo_name": repo.name,
        "repo_path": str(repo),
        "cutoff_date": cutoff_iso,
        "cutoff_default_head": default_heads,
        "cutoff_all_heads": all_heads,
        "first_commit_date": first_commit_date,
        "timeline_scope": TIMELINE_SCOPE,
        "timeline": timeline,
        "events": events,
        "snapshot_as_of_cutoff_default": snapshot_default,
        "snapshot_as_of_cutoff_repo_wide": snapshot_repo_wide,
        "off_default_extra_styles": off_default_extra,
        "has_off_default_extra": int(bool(off_default_extra)),
        "empty_gaps": empty_gaps,
        **({"qa_issue": qa_issue} if qa_issue else {}),
    }

# ===========================
# Runner + index/summary
# ===========================
def output_path_for(repo: Path) -> Path:
    return OUTPUT_MINE / f"{repo.name}.emulator_timeline.json"

def list_repos_once(root: Path) -> List[Path]:
    rs = list_repos(root)
    rs.sort(key=lambda p: p.name.lower())
    return rs

def iso_min(dts: List[str]) -> Optional[str]:
    ds = [d for d in dts if d]
    return min(ds) if ds else None

def derive_summary_fields(rec: Dict) -> Dict:
    name = rec.get("repo_name")
    snap_def = rec.get("snapshot_as_of_cutoff_default", [])
    snap_all = rec.get("snapshot_as_of_cutoff_repo_wide", [])
    events = rec.get("events", {})
    first_events: List[Optional[str]] = []
    for k in ("Emu_Community", "Emu_Custom", "GMD", "ThirdParty"):
        for e in events.get(k, []):
            if e.get("event") == "added":
                first_events.append(e.get("date"))
    first_env_date = iso_min([d for d in first_events if d])
    transitions = 0
    for k in events:
        transitions += sum(1 for e in events[k] if e.get("event") in ("added", "removed"))
    gaps = rec.get("empty_gaps", [])
    gap_days = 0.0
    for g in gaps:
        try:
            gap_days += float(g.get("duration_days") or 0.0)
        except Exception:
            pass
    return {
        "repo_name": name,
        "has_env_at_cutoff_default": 1 if snap_def else 0,
        "has_env_at_cutoff_repo_wide": 1 if snap_all else 0,
        "cutoff_states_default": "+".join(snap_def),
        "cutoff_states_repo_wide": "+".join(snap_all),
        "off_default_extra": "+".join(rec.get("off_default_extra_styles", [])),
        "has_off_default_extra": int(bool(rec.get("off_default_extra_styles"))),
        "first_env_date": first_env_date or "",
        "transitions": transitions,
        "total_gap_days": round(gap_days, 2),
        "qa_issue": rec.get("qa_issue", ""),
    }

def run_miner(cutoff_iso: str = CUTOFF_ISO, max_repos: int = MAX_REPOS) -> None:
    OUTPUT_MINE.mkdir(parents=True, exist_ok=True)

    repos = list_repos_once(INPUT_CLONES)
    if max_repos and max_repos > 0:
        repos = repos[:max_repos]

    if not repos:
        print(f"[info] No git repos found under: {INPUT_CLONES}")
        return

    print(f"[info] Work root:         {WORK_ROOT}")
    print(f"[info] Input repos:       {INPUT_CLONES}")
    print(f"[info] Output JSON dir:   {OUTPUT_MINE}")
    print(f"[info] Found repos:       {len(repos)}")
    print(f"[info] Cutoff (UTC):      {cutoff_iso}")
    print(f"[info] Timeline scope:    {TIMELINE_SCOPE}")
    print(f"[info] Options: suppress_empty_rows={SUPPRESS_EMPTY_ROWS}, capture_empty_gaps={CAPTURE_EMPTY_GAPS}, emit_none={EMIT_NONE_STATE}, blob_max_size={BLOB_MAX_SIZE}")

    written = skipped = errors = 0
    summaries: List[Dict] = []
    env_rows: List[Dict] = []

    

    def process(repo: Path) -> Tuple[str, Path, Optional[Dict], Optional[str]]:
        try:
            outp = output_path_for(repo)
            if RESUME_IF_EXISTS and outp.exists():
                return ("skipped", repo, None, None)
            data = build_timeline(repo, cutoff_iso)
            with outp.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            return ("written", repo, data, None)
        except Exception as e:
            return ("error", repo, None, str(e))

    with PoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(process, r): r for r in repos}
        total = len(futs)
        for i, fut in enumerate(as_completed(futs), 1):
            repo = futs[fut]
            rel = repo.relative_to(INPUT_CLONES) if repo != INPUT_CLONES else Path(repo.name)
            status, _repo, data, err = fut.result()
            if status == "written":
                print(f"[{i}/{total}] Wrote: {rel}")
                written += 1
                summaries.append(derive_summary_fields(data))  # type: ignore[arg-type]
                # NEW: build env/PR/issue rows from the freshly computed data
                env_rows.extend(_build_env_event_rows_for_repo(repo, data))  # type: ignore[arg-type]

            elif status == "skipped":
                print(f"[{i}/{total}] Skipped (exists): {rel}")
                skipped += 1
                try:
                    with output_path_for(repo).open("r", encoding="utf-8") as f:
                        data2 = json.load(f)
                    summaries.append(derive_summary_fields(data2))
                    # NEW: also build rows from existing JSON
                    env_rows.extend(_build_env_event_rows_for_repo(repo, data2))
                except Exception:
                    pass

            else:
                print(f"[i/{total}] [error] {rel}: {err}", file=sys.stderr)
                errors += 1


    # Write summary CSV + JSON index
    idx_json = OUTPUT_MINE / "index_summary.json"
    with idx_json.open("w", encoding="utf-8") as f:
        json.dump({"cutoff": cutoff_iso, "repos": summaries}, f, ensure_ascii=False, indent=2)

    idx_csv = OUTPUT_MINE / "index_summary.csv"
    with idx_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "repo_name",
                "has_env_at_cutoff_default",
                "has_env_at_cutoff_repo_wide",
                "cutoff_states_default",
                "cutoff_states_repo_wide",
                "off_default_extra",
                "has_off_default_extra",
                "first_env_date",
                "transitions",
                "total_gap_days",
                "qa_issue",
            ],
        )
        w.writeheader()
        for row in summaries:
            w.writerow(row)

    qa_csv = OUTPUT_MINE / "qa_issues.csv"
    with qa_csv.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["repo_name", "qa_issue"])
        w.writeheader()
        for row in summaries:
            if row.get("qa_issue"):
                w.writerow({"repo_name": row["repo_name"], "qa_issue": row["qa_issue"]})

    print(f"[done] JSON -> {OUTPUT_MINE}  (written={written}, skipped={skipped}, errors={errors})")
    print(f"[done] Summary: {idx_csv.name}, {idx_json.name}, QA: {qa_csv.name}")

        # NEW: write per-event env + PR + issues CSV
    env_csv = OUTPUT_MINE / "env_events_with_pr_issues.csv"
    if env_rows:
        fieldnames = [
            "full_name",
            "repo_folder",
            "env_style",
            "change_type",
            "change_date",
            "on_default",
            "commit",
            "commit_short",
            "commit_subject",
            "commit_body",
            "pr_number",
            "pr_state",
            "pr_title",
            "pr_body",
            "pr_created_at",
            "pr_merged_at",
            "pr_closed_at",
            "pr_user_login",
            "pr_labels",
            "pr_html_url",
            "issues_summary",
        ]
        with env_csv.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            for row in env_rows:
                w.writerow(row)
        print(f"[done] Env/PR/Issues CSV: {env_csv.name}")
    else:
        print("[info] No env events found to write to env_events_with_pr_issues.csv")


if __name__ == "__main__":
    _failfast_checks()
    print("[run] Starting miner…")
    run_miner(cutoff_iso=CUTOFF_ISO, max_repos=MAX_REPOS)
    print("[run] Finished.")


[run] Starting miner…
[info] Work root:         C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4
[info] Input repos:       C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\clonesV1.0
[info] Output JSON dir:   C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\mineV1.0
[info] Found repos:       1
[info] Cutoff (UTC):      2025-08-10 23:59:59 +0000
[info] Timeline scope:    repo_wide
[info] Options: suppress_empty_rows=True, capture_empty_gaps=True, emit_none=False, blob_max_size=2000000
[1/1] Wrote: connectbot__connectbot
[done] JSON -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\mineV1.0  (written=1, skipped=0, errors=0)
[done] Summary: index_summary.csv, index_summary.json, QA: qa_issues.csv
[done] Env/PR/Issues CSV: env_events_with_pr_issues.csv
[run] Finished.


## Episodes defintion: Start to current timeline

In [12]:
# === Build dataset for Observation 3.1 (V9.3 aligned) ===
# CHANGES:
# - DROP current_*_strict columns (they were misleading for your use case)
# - "FIRST" now uses a per-style union rule:
#     The earliest style to achieve ≥14 days of continuous coverage (no projecting beyond cutoff).
#   -> Updates: first_date, first_year, first_label, first_styleset
#   -> Also writes: first_styleset_union14d (same value as first_styleset for clarity)
# - "TIMELINE FORMATION" styles:
#     timeline_styleset_union14d = all styles that EVER achieved ≥14 days at any time
#     (filters out temporary (<14d) add+remove changes; does not use post-cutoff time)
# - "CURRENT" stays snapshot-based (repo-wide inventory at cutoff, no ≥14d):
#     current_label_snapshot, current_styleset_snapshot
# - Default-branch snapshot kept as current_styleset_default_B
# - num_episodes counts episodes with duration ≥14 days (as before)
#
# NEW:
# - Also builds a per-event CSV:
#   OBS_OUTPUT / "obs3_1_change_events_commits_basic.csv"
#   One row per env event (added/removed/maintenance) per style, per repo, including:
#   - commit SHA
#   - event date (raw + UTC)
#   - style, event_type, on_default
#   - linkage to ≥14-day episodes:
#       in_persistent_episode, episode_index,
#       episode_start_utc, episode_end_utc, episode_duration_days,
#       is_episode_start_boundary, is_episode_end_boundary

from __future__ import annotations
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict

import pandas as pd

# ---- Paths ----
WORK_ROOT   = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4")
OUTPUT_MINE = WORK_ROOT / "mineV1.0"

MAIN_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\5.0_Total_Repo.csv")

OBS_OUTPUT = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations")
OBS_OUTPUT.mkdir(parents=True, exist_ok=True)

# ---- Settings ----
CUTOFF_ISO   = "2025-08-10 23:59:59 +0000"
PERSIST_DAYS = 14.0
CANONICAL_STYLES: Set[str] = {"Emu_Community", "Emu_Custom", "GMD", "ThirdParty"}

# ---- Helpers ----
def parse_iso(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    s = s.strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    # normalize +0000 -> +00:00
    if len(s) >= 5 and (s[-5] in ["+","-"]) and s[-3] != ":":
        s = s[:-2] + ":" + s[-2:]
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None

CUTOFF_DT = parse_iso(CUTOFF_ISO)
assert CUTOFF_DT is not None, f"Bad CUTOFF_ISO: {CUTOFF_ISO}"

def days_between(a: datetime, b: datetime) -> float:
    return (b - a).total_seconds() / 86400.0

def normalize_styles(style_list: List[str]) -> Set[str]:
    return {s for s in (style_list or []) if s in CANONICAL_STYLES}

def label_from_set(s: Set[str]) -> str:
    if "Emu_Community" in s and "Emu_Custom" in s:
        return "Both"
    if "Emu_Community" in s:
        return "Community"
    if "Emu_Custom" in s:
        return "Custom"
    if "GMD" in s:
        return "GMD"
    if "ThirdParty" in s:
        return "ThirdParty"
    return "None"

def read_repo_jsons(output_dir: Path) -> List[Path]:
    return sorted([p for p in output_dir.glob("*.emulator_timeline.json")])

def build_episodes(timeline_rows: List[Dict], cutoff_dt: datetime) -> List[Dict]:
    """Build contiguous episodes from timeline rows with non-empty canonical styles, capped at cutoff."""
    rows = [r for r in (timeline_rows or []) if r.get("styles")]
    rows.sort(key=lambda r: parse_iso(r.get("date")) or datetime.min.replace(tzinfo=timezone.utc))

    episodes: List[Dict] = []
    for i, r in enumerate(rows):
        start = parse_iso(r.get("date"))
        if start is None:
            continue

        styles_now = normalize_styles(list(r.get("styles") or []))
        if not styles_now:
            continue

        # end at the date of the next canonical row, else at cutoff
        end: Optional[datetime] = None
        for j in range(i + 1, len(rows)):
            nxt = rows[j]
            nxt_styles = normalize_styles(list(nxt.get("styles") or []))
            if nxt_styles:
                end = parse_iso(nxt.get("date"))
                break
        if end is None:
            end = cutoff_dt
        if end <= start:
            continue

        episodes.append({
            "start": start,
            "end": min(end, cutoff_dt),
            "styles": styles_now,
            "duration_days": max(0.0, days_between(start, min(end, cutoff_dt))),
        })
    return episodes

# ---- Per-style interval utilities ----
def merge_intervals(intervals: List[Tuple[datetime, datetime]]) -> List[Tuple[datetime, datetime]]:
    """Merge overlapping/contiguous [start,end] intervals."""
    if not intervals:
        return []
    intervals.sort(key=lambda ab: ab[0])
    merged: List[Tuple[datetime, datetime]] = []
    cur_s, cur_e = intervals[0]
    for a, b in intervals[1:]:
        if a <= cur_e:  # overlap or contiguous
            if b > cur_e: cur_e = b
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = a, b
    merged.append((cur_s, cur_e))
    return merged

def per_style_intervals(episodes: List[Dict]) -> Dict[str, List[Tuple[datetime, datetime]]]:
    """Intervals per style drawn from episode styles (episodes already capped at cutoff)."""
    m: Dict[str, List[Tuple[datetime, datetime]]] = {s: [] for s in CANONICAL_STYLES}
    for ep in episodes:
        for s in (ep["styles"] or []):
            if s in CANONICAL_STYLES:
                m[s].append((ep["start"], ep["end"]))
    # merge
    return {s: merge_intervals(v) for s, v in m.items() if v}

# ---- FIRST via union ≥14d ----
def first_union14d(episodes: List[Dict], min_days: float) -> Tuple[Optional[datetime], Optional[str]]:
    """
    Earliest moment any single style achieves ≥min_days of continuous coverage.
    Returns (first_date, style_name) or (None, None).
    """
    psi = per_style_intervals(episodes)
    best_date: Optional[datetime] = None
    best_style: Optional[str] = None

    # deterministic style tie-break order
    order = ["Emu_Community", "Emu_Custom", "GMD", "ThirdParty"]
    for style in order:
        if style not in psi: 
            continue
        for a, b in psi[style]:
            if days_between(a, b) >= min_days:
                t = a + timedelta(days=min_days)
                if best_date is None or t < best_date or (t == best_date and order.index(style) < order.index(best_style)):  # type: ignore
                    best_date = t
                    best_style = style
                break  # earliest interval for this style is enough

    return best_date, best_style

# ---- TIMELINE formation set (union of styles that ever achieved ≥14d) ----
def timeline_union14d_styles(episodes: List[Dict], min_days: float) -> Set[str]:
    psi = per_style_intervals(episodes)
    kept: Set[str] = set()
    for s, ivs in psi.items():
        for a, b in ivs:
            if days_between(a, b) >= min_days:
                kept.add(s)
                break
    return kept

def _find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lowmap = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lowmap:
            return lowmap[name.lower()]
    return None

# ---- Build OBS core from miner ----
records: List[Dict] = []

# NEW: per-event rows across all repos
episode_event_rows: List[Dict] = []

json_files = read_repo_jsons(OUTPUT_MINE)
if not json_files:
    raise FileNotFoundError(f"No miner JSON files found under: {OUTPUT_MINE}")

for jf in json_files:
    with jf.open("r", encoding="utf-8") as f:
        rec = json.load(f)

    repo = rec.get("repo_name") or jf.stem
    full_name = (repo or "").replace("__", ".")
    timeline = rec.get("timeline") or []

    episodes = build_episodes(timeline, CUTOFF_DT)

    # --- EPISODES: only those with duration ≥14 days are "persistent" episodes ---
    persistent_eps = [ep for ep in episodes if ep.get("duration_days", 0.0) >= PERSIST_DAYS]
    # assign stable indexes based on time
    persistent_eps.sort(key=lambda ep: ep["start"])
    for idx, ep in enumerate(persistent_eps, start=1):
        ep["episode_index"] = idx

    # Precompute episode boundary maps: (style, timestamp_utc) -> [episode_index,...]
    start_boundaries = defaultdict(list)
    end_boundaries = defaultdict(list)
    for ep in persistent_eps:
        start_key = ep["start"].astimezone(timezone.utc).isoformat()
        end_key   = ep["end"].astimezone(timezone.utc).isoformat()
        for s in ep["styles"]:
            if s in CANONICAL_STYLES:
                start_boundaries[(s, start_key)].append(ep["episode_index"])
                end_boundaries[(s, end_key)].append(ep["episode_index"])

    # --- Build per-event rows for THIS repo ---
    events_dict = rec.get("events") or {}
    for style in CANONICAL_STYLES:
        for ev in events_dict.get(style, []):
            ev_type = ev.get("event", "")
            raw_date = ev.get("date", "")
            dt = parse_iso(raw_date)
            if dt is None:
                continue
            dt_utc = dt.astimezone(timezone.utc)
            dt_key = dt_utc.isoformat()

            # map this event into (at most one) persistent episode for this style
            in_persist = 0
            ep_idx = None
            ep_start = ep_end = None
            ep_dur = None

            for ep in persistent_eps:
                # episode is active if style is in its styles and date in [start,end]
                if style in ep["styles"] and ep["start"] <= dt_utc <= ep["end"]:
                    in_persist = 1
                    ep_idx = ep.get("episode_index")
                    ep_start = ep["start"].astimezone(timezone.utc).isoformat()
                    ep_end = ep["end"].astimezone(timezone.utc).isoformat()
                    ep_dur = ep.get("duration_days")
                    break

            is_start_boundary = 1 if (style, dt_key) in start_boundaries else 0
            is_end_boundary   = 1 if (style, dt_key) in end_boundaries else 0

            episode_event_rows.append({
                "repo_name": repo,
                "full_name": full_name,
                "env_style": style,
                "event_type": ev_type,
                "event_date_raw": raw_date,
                "event_date_utc": dt_key,
                "commit": ev.get("commit", ""),
                "on_default": ev.get("on_default", 0),

                "in_persistent_episode": in_persist,
                "episode_index": ep_idx if ep_idx is not None else "",
                "episode_start_utc": ep_start if ep_start is not None else "",
                "episode_end_utc": ep_end if ep_end is not None else "",
                "episode_duration_days": ep_dur if ep_dur is not None else "",

                "is_episode_start_boundary": is_start_boundary,
                "is_episode_end_boundary": is_end_boundary,
            })

    # --- Summary metrics per repo as before ---
    # Count episodes with ≥14d (for reference)
    num_episodes = len(persistent_eps)

    # FIRST via union ≥14d
    first_dt, first_style = first_union14d(episodes, PERSIST_DAYS)
    if first_dt and first_style:
        first_styleset_union14d = first_style
        first_label = label_from_set({first_style})
        first_year = first_dt.year
    else:
        first_styleset_union14d = ""
        first_label = "None"
        first_year = ""

    # TIMELINE formation set (filters out temporary <14d styles)
    timeline_set = timeline_union14d_styles(episodes, PERSIST_DAYS)
    timeline_styleset_union14d = "+".join(sorted(timeline_set)) if timeline_set else ""
    timeline_label_union14d = label_from_set(timeline_set)

    # CURRENT snapshot (repo-wide, no ≥14d)
    snap_def_raw  = rec.get("snapshot_as_of_cutoff_default") or []
    snap_all_raw  = rec.get("snapshot_as_of_cutoff_repo_wide") or []
    snap_def_norm_set  = normalize_styles(snap_def_raw)
    snap_all_norm_set  = normalize_styles(snap_all_raw)

    current_label_snapshot    = label_from_set(snap_all_norm_set)
    current_styleset_snapshot = "+".join(sorted(snap_all_norm_set)) if snap_all_norm_set else ""

    # Default-branch snapshot
    current_styleset_default_B = "+".join(sorted(snap_def_norm_set)) if snap_def_norm_set else ""

    # Optional diagnostic: short-and-active-at-cutoff flag (kept)
    recent_lt14d = 0
    if episodes:
        last_ep = episodes[-1]
        if last_ep["end"] >= CUTOFF_DT and last_ep["duration_days"] < PERSIST_DAYS:
            recent_lt14d = 1

    out = {
        "repo_name": repo,
        "full_name": full_name,
        "cutoff_date": CUTOFF_ISO,

        "num_episodes": num_episodes,

        # FIRST (union ≥14d)
        "first_date": first_dt.isoformat() if first_dt else "",
        "first_year": first_year,
        "first_label": first_label,
        "first_styleset": first_styleset_union14d,           # updated meaning
        "first_styleset_union14d": first_styleset_union14d,  # explicit column

        # TIMELINE formation set (ever ≥14d)
        "timeline_styleset_union14d": timeline_styleset_union14d,
        "timeline_label_union14d": timeline_label_union14d,

        # CURRENT (snapshot)
        "current_label_snapshot": current_label_snapshot,
        "current_styleset_snapshot": current_styleset_snapshot,
        "current_styleset_default_B": current_styleset_default_B,

        # Diagnostics
        "current_recent_lt14d": recent_lt14d,
        "snapshot_default_norm": "+".join(sorted(snap_def_norm_set)) if snap_def_norm_set else "",
        "snapshot_repo_wide_norm": "+".join(sorted(snap_all_norm_set)) if snap_all_norm_set else "",
        "qa_issue": rec.get("qa_issue", ""),
    }
    records.append(out)

# ==== Build repo-level summary DF (same as before) ====
df = pd.DataFrame.from_records(records).sort_values(["first_date", "repo_name"], na_position="last")

# ---- Merge MAIN: include 'execution_environment' as-is; exclude api_*; prefix the rest with main_ ----
if MAIN_CSV.exists():
    df_main = pd.read_csv(MAIN_CSV, dtype=str, encoding="utf-8", keep_default_na=False)

    col_full = _find_col(df_main, ["full_name", "Full_name", "repo_full_name"])
    col_exec = _find_col(df_main, ["execution_environment", "Execution_Environment"])

    if col_full:
        df["_key"] = df["full_name"].astype(str).str.lower().str.strip()
        df_main["_key"] = df_main[col_full].astype(str).str.lower().str.strip()

        # execution_environment as-is
        if col_exec:
            df_exec = df_main[["_key", col_exec]].copy()
            df_exec.rename(columns={col_exec: "execution_environment"}, inplace=True)
            df = df.merge(df_exec, on="_key", how="left")
        else:
            df["execution_environment"] = ""

        # other MAIN columns (exclude join col, exec col, any api_*)
        main_cols = []
        for c in df_main.columns:
            cl = c.lower()
            if c in {"_key", col_full}: continue
            if col_exec and c == col_exec: continue
            if cl.startswith("api_"): continue
            main_cols.append(c)
        if main_cols:
            df_other = df_main[["_key"] + main_cols].copy()
            df_other.rename(columns={c: f"main_{c}" for c in main_cols}, inplace=True)
            df = df.merge(df_other, on="_key", how="left")

        df.drop(columns=["_key"], inplace=True, errors="ignore")
    else:
        df["execution_environment"] = ""
else:
    df["execution_environment"] = ""

# ---- Column order: repo_name, full_name, execution_environment, then the rest ----
cols = list(df.columns)
for required in ["repo_name", "full_name", "execution_environment"]:
    if required not in cols:
        cols.insert(0, required)

# place full_name after repo_name
if "repo_name" in cols and "full_name" in cols:
    cols.remove("full_name")
    cols.insert(cols.index("repo_name") + 1, "full_name")

# place execution_environment after full_name
if "execution_environment" in cols:
    cols.remove("execution_environment")
    insert_at = cols.index("full_name") + 1 if "full_name" in cols else 1
    cols.insert(insert_at, "execution_environment")

df = df[cols]

# ---- FILTER: exclude repos where first_label == "None" ----
pre_filter_n = len(df)
df = df[df["first_label"] != "None"].copy()
post_filter_n = len(df)
dropped_n = pre_filter_n - post_filter_n
print(f"[info] Excluding repos with first_label == 'None': dropped {dropped_n} of {pre_filter_n}")

# ---- Save repo-level summary ----
out_csv = OBS_OUTPUT / "obs3_1_entry_vs_current.csv"
df.to_csv(out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote dataset (filtered): {out_csv}")

# ---- Save event-level / commit-level CSV for episode analysis ----
# ---- Save event-level / commit-level CSV for episode analysis ----
events_df = pd.DataFrame.from_records(episode_event_rows)

if not events_df.empty:
    # Keep ONLY the commits that actually create the ≥14-day change episodes:
    #  - events inside a persistent episode
    #  - event_type is added/removed (no maintenance)
    #  - event is at start or end boundary of that persistent episode
    mask = (
        (events_df["in_persistent_episode"] == 1) &
        (events_df["event_type"].isin(["added", "removed"])) &
        (
            (events_df["is_episode_start_boundary"] == 1) |
            (events_df["is_episode_end_boundary"] == 1)
        )
    )
    events_df = events_df[mask].copy()

    # Nice ordering
    sort_cols = [
        "repo_name",
        "episode_index",
        "event_date_utc",
        "env_style",
        "event_type",
    ]
    sort_cols = [c for c in sort_cols if c in events_df.columns]
    if sort_cols:
        events_df.sort_values(sort_cols, inplace=True)

events_out_csv = OBS_OUTPUT / "obs3_1_change_episodes_commits.csv"
events_df.to_csv(events_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote episode-boundary commit dataset: {events_out_csv}")


# Optional quick peek
try:
    show_cols = [
        "repo_name","full_name","execution_environment",
        "num_episodes",
        "first_year","first_label","first_date",
        "first_styleset","first_styleset_union14d",
        "timeline_styleset_union14d","timeline_label_union14d",
        "current_label_snapshot","current_styleset_snapshot",
        "current_styleset_default_B",
        "current_recent_lt14d",
        "snapshot_repo_wide_norm"
    ]
    show_cols += [c for c in df.columns if c.startswith("main_")][:3]
    print(df[show_cols].head(20).to_string(index=False))
except Exception:
    pass


[info] Excluding repos with first_label == 'None': dropped 0 of 1
[ok] Wrote dataset (filtered): C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations\obs3_1_entry_vs_current.csv
[ok] Wrote episode-boundary commit dataset: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations\obs3_1_change_episodes_commits.csv
             repo_name             full_name execution_environment  num_episodes  first_year first_label                first_date first_styleset first_styleset_union14d timeline_styleset_union14d timeline_label_union14d current_label_snapshot current_styleset_snapshot current_styleset_default_B  current_recent_lt14d snapshot_repo_wide_norm
connectbot__connectbot connectbot.connectbot                                   2        2016      Custom 2016-06-17T04:31:19+00:00     Emu_Custom              Emu_Custom   Emu_Community+Emu_Custom                    Both              Community             Emu_Community              Emu_Community                 